# Cohort B: healthcare-utilization "density" features

Mirrors `05_EDI_Feature_set_up_cohort_A`, applied to Cohort B. Same four steps: raw utilization
metrics per patient-year, GMM density clustering (scored against the same pre-fit bundle used for
Cohort A), per-cluster documentation-volume residuals, and a final join onto the episode-level
feature table. Produces `cohort_b_derived.cohort_b_feature_table_edi`, used downstream in
`08_union_cohort_a_and_b`.

In [ ]:
%sql
USE CATALOG
your_catalog;

### Step 1 — raw utilization metrics per patient-year
Same logic as Cohort A: visit counts, inpatient stays, visit-gap regularity, and
condition/drug/procedure/measurement counts. Writes `cohort_b_derived.cohort_b_edi_features`.

In [ ]:
%python
#feature table has: visit_count, inpatient_visit_count,
# hospitalized_days, irregularity_l1, irregularity_l2, has_valid_regularity

# COMMAND ----------

import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, DoubleType

# OMOP standard concept ids for inpatient visit types (9201 = Inpatient Visit, 262 = ER + Inpatient Visit)
INPATIENT_VISIT_CONCEPT_IDS = [9201, 262]

# COMMAND ----------

# Regularity metric: mean-gap-normalized L1/L2 variation of inter-visit intervals
def compute_regularity(gaps):
    if gaps is None or len(gaps) <= 1:
        return (None, None, None)
    gaps = np.array(gaps)
    mean_gap = np.mean(gaps)
    if mean_gap <= 1e-10:
        return (None, None, None)
    irregularity_l1 = float(np.mean(np.abs((gaps / mean_gap) - 1)))
    irregularity_l2 = float(np.sqrt(np.mean((gaps / mean_gap - 1) ** 2)))
    return (irregularity_l1, irregularity_l2, float(mean_gap))


regularity_schema = StructType([
    StructField("irregularity_l1", DoubleType(), True),
    StructField("irregularity_l2", DoubleType(), True),
    StructField("central_diff", DoubleType(), True),
])
compute_regularity_udf = F.udf(compute_regularity, regularity_schema)

# COMMAND ----------


visit_daily = (
    spark.table("cohort_b.visit_occurrence")
    .select(
        F.col("person_id"),
        F.col("visit_occurrence_id"),
        F.col("visit_concept_id"),
        F.to_date("visit_start_date").alias("start_date"),
        F.to_date("visit_end_date").alias("end_date"),
    )
    .filter(F.col("start_date").isNotNull())
    .withColumn(
        "end_date",
        F.when(F.col("end_date").isNull() | (F.col("end_date") < F.col("start_date")), F.col("start_date"))
        .otherwise(F.col("end_date")),
    )
    .withColumn("date", F.explode(F.expr("sequence(start_date, end_date, interval 1 day)")))
    .withColumn("is_hospitalized", F.when(F.col("visit_concept_id").isin(*INPATIENT_VISIT_CONCEPT_IDS), 1).otherwise(0))
)

# COMMAND ----------

# Collapse to one row per person-day: distinct visit count, visit ids (for dedupe later), hospitalized flag
daily = (
    visit_daily.groupBy("person_id", "date")
    .agg(
        F.countDistinct("visit_occurrence_id").alias("visit_count"),
        F.collect_set("visit_occurrence_id").alias("visit_occurrence_ids"),
        F.max("is_hospitalized").alias("hospitalized_flag"),
    )
    .withColumn("year", F.year("date"))
    .repartition("person_id")
    .cache()
)
row_count = daily.count()
print(f"Loaded {row_count:,} person-days from visit_occurrence")

# COMMAND ----------

# Core feature 1: visit_count
visit_frequency = daily.groupBy("person_id", "year").agg(
    F.sum("visit_count").alias("visit_count")
)

# COMMAND ----------

# Core features 2-3: inpatient_visit_count, hospitalized_days
hospitalization = (
    daily.groupBy("person_id", "year")
    .agg(
        F.size(
            F.array_distinct(
                F.flatten(
                    F.collect_list(F.when(F.col("hospitalized_flag") == 1, F.col("visit_occurrence_ids")))
                )
            )
        ).alias("inpatient_visit_count"),
        F.countDistinct(F.when(F.col("hospitalized_flag") == 1, F.col("date"))).alias("hospitalized_days"),
    )
    .fillna(0, subset=["inpatient_visit_count", "hospitalized_days"])
)

# COMMAND ----------

# Core features 4-6: irregularity_l1, irregularity_l2, has_valid_regularity
patient_date_window = Window.partitionBy("person_id", "year").orderBy("date")

active_days = (
    daily.filter(F.col("visit_count") > 0)
    .withColumn("prev_date", F.lag("date", 1).over(patient_date_window))
    .withColumn(
        "days_between_visits",
        F.when(F.col("prev_date").isNotNull(), F.datediff("date", "prev_date")),
    )
)

gaps_collected = (
    active_days.filter(F.col("days_between_visits").isNotNull())
    .groupBy("person_id", "year")
    .agg(F.collect_list("days_between_visits").alias("gaps_list"))
)

regularity = gaps_collected.withColumn(
    "regularity_metrics", compute_regularity_udf(F.col("gaps_list"))
).select(
    "person_id",
    "year",
    F.col("regularity_metrics.irregularity_l1").alias("irregularity_l1"),
    F.col("regularity_metrics.irregularity_l2").alias("irregularity_l2"),
)

condition_counts = (
    spark.table("cohort_b.condition_occurrence")
    .withColumn("year", F.year("condition_start_date"))
    .groupBy("person_id", "year")
    .agg(F.count("*").alias("total_condition_count"))
)

drug_counts = (
    spark.table("cohort_b.drug_exposure")
    .withColumn("year", F.year("drug_exposure_start_date"))
    .groupBy("person_id", "year")
    .agg(F.count("*").alias("total_drug_count"))
)

procedure_counts = (
    spark.table("cohort_b.procedure_occurrence")
    .withColumn("year", F.year("procedure_date"))
    .groupBy("person_id", "year")
    .agg(F.count("*").alias("total_procedure_count"))
)

measurement_counts = (
    spark.table("cohort_b.measurement")
    .withColumn("year", F.year("measurement_date"))
    .groupBy("person_id", "year")
    .agg(F.count("*").alias("total_measurement_count"))
)


# COMMAND ----------

# Assemble + save
features = (
    visit_frequency.join(hospitalization, ["person_id", "year"], "left")
    .join(regularity, ["person_id", "year"], "left")
    .fillna(0, subset=["visit_count", "inpatient_visit_count"])
    .withColumn(
        "has_valid_regularity",
        F.when(F.col("irregularity_l1").isNotNull(), 1).otherwise(0),
    )
)

features = (
    features
    .join(condition_counts, ["person_id", "year"], "left")
    .join(drug_counts, ["person_id", "year"], "left")
    .join(procedure_counts, ["person_id", "year"], "left")
    .join(measurement_counts, ["person_id", "year"], "left")
    .fillna(0, subset=["total_condition_count", "total_drug_count", "total_procedure_count", "total_measurement_count"])
)

features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "cohort_b_derived.cohort_b_edi_features"
)
print(f"Wrote {features.count():,} person-years")

daily.unpersist()

### Step 2 — utilization density clustering
Scores Cohort B patient-years against the same pre-fit GMM bundle used for Cohort A. Writes
`cohort_b_derived.gmm_4_density_scores`.

In [ ]:
%python

import os
import numpy as np
import pandas as pd
import cloudpickle
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

INPUT_TABLE = "cohort_b_derived.cohort_b_edi_features"
MODE = "score"
BUNDLE_DIR = "/Volumes/your_catalog/cohort_a_derived/gmm_datadensity"  
BUNDLE_PATH =  f"{BUNDLE_DIR}/gmm_bundle.pkl"
OUTPUT_TABLE = "cohort_b_derived.gmm_4_density_scores"

OVARIANCE_TYPE = "tied"
N_INIT = 10
MAX_ITER = 200
RANDOM_STATE = 42
K = 4

# ---- Core feature list + label map (do not reorder) ----
CORE_FEATURES = [
    "visit_count",
    "inpatient_visit_count",
    "hospitalized_days",
    "has_valid_regularity",
    "irregularity_l1",
    "irregularity_l2",
]
WINSORIZE_FEATURES = ["visit_count", "inpatient_visit_count", "hospitalized_days"]
GMM4_LABELS = {
    0: "Moderate Inpatient",
    1: "Outpatient Irregular",
    2: "Outpatient Regular",
    3: "High Inpatient",
}

# ---- Preprocessing pipeline ----
def prepare_feature_matrix(features_df, fit_scaler, scaler=None, winsor_caps=None, percentile=0.99, seed=RANDOM_STATE):
    features_df = features_df.copy()
    features_df["irregularity_l1"] = features_df["irregularity_l1"].fillna(0.0)
    features_df["irregularity_l2"] = features_df["irregularity_l2"].fillna(0.0)

    feature_matrix = np.nan_to_num(features_df[CORE_FEATURES].values, nan=0.0, posinf=0.0, neginf=0.0)

    if winsor_caps is None:
        winsor_caps = {}
        for i, feat in enumerate(CORE_FEATURES):
            if feat in WINSORIZE_FEATURES:
                col_vals = feature_matrix[:, i]
                winsor_caps[feat] = float(np.nanpercentile(col_vals[np.isfinite(col_vals)], percentile * 100))

    for i, feat in enumerate(CORE_FEATURES):
        if feat in WINSORIZE_FEATURES and feat in winsor_caps:
            cap = winsor_caps[feat]
            if np.isfinite(cap):
                feature_matrix[:, i] = np.minimum(feature_matrix[:, i], cap)

    feature_matrix_log = feature_matrix.copy()
    for i, feat in enumerate(CORE_FEATURES):
        if feat in WINSORIZE_FEATURES:
            feature_matrix_log[:, i] = np.log1p(feature_matrix_log[:, i])

    if fit_scaler or scaler is None:
        scaler = StandardScaler()
        feature_matrix_scaled = scaler.fit_transform(feature_matrix_log)
    else:
        feature_matrix_scaled = scaler.transform(feature_matrix_log)

    np.random.seed(seed)
    feature_matrix_scaled += np.random.normal(0, 1e-10, feature_matrix_scaled.shape)

    return feature_matrix_scaled, scaler, winsor_caps

# ---- Load features ----
features = spark.table(INPUT_TABLE).select("person_id", "year", *CORE_FEATURES).cache()
row_count = features.count()
print(f"Loaded {row_count:,} person-years from {INPUT_TABLE}")

# ---- Score or train ----
if MODE == "score":
    with open(BUNDLE_PATH, "rb") as f:
        bundle = cloudpickle.load(f)
    gmm_model = bundle.get("model") or bundle.get("gmm_model")
    scaler = bundle["scaler"]
    winsor_caps = bundle["winsor_caps"]
    if gmm_model is None:
        raise ValueError(f"Bundle at {BUNDLE_PATH} is missing a 'model' entry")
    print(f"Loaded pretrained bundle from {BUNDLE_PATH}")

    features_pdf = features.toPandas()
    feature_matrix_scaled, _, _ = prepare_feature_matrix(features_pdf, fit_scaler=False, scaler=scaler, winsor_caps=winsor_caps)

else:
    person_window = Window.partitionBy("person_id").orderBy(F.rand(seed=RANDOM_STATE))
    training_sample_ids = (
        features.select("person_id", "year").distinct()
        .withColumn("row_num", F.row_number().over(person_window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    features_pdf = features.toPandas()
    sample_pdf = features_pdf.merge(training_sample_ids.toPandas(), on=["person_id", "year"])

    train_matrix, scaler, winsor_caps = prepare_feature_matrix(sample_pdf, fit_scaler=True)

    print(f"Training GMM-4 on {len(sample_pdf):,} stratified person-years...")
    gmm_model = GaussianMixture(
        n_components=K, covariance_type=COVARIANCE_TYPE, n_init=N_INIT,
        max_iter=MAX_ITER, random_state=RANDOM_STATE, verbose=1,
    )
    gmm_model.fit(train_matrix)
    print(f"Converged={gmm_model.converged_}, BIC={gmm_model.bic(train_matrix):.2f}")

    feature_matrix_scaled, _, _ = prepare_feature_matrix(features_pdf, fit_scaler=False, scaler=scaler, winsor_caps=winsor_caps)

    os.makedirs(BUNDLE_DIR, exist_ok=True)
    with open(BUNDLE_PATH, "wb") as f:
        cloudpickle.dump({"model": gmm_model, "scaler": scaler, "winsor_caps": winsor_caps}, f)
    print(f"Saved bundle to {BUNDLE_PATH}")

# ---- Predict cluster, confidence, entropy ----
cluster_labels = gmm_model.predict(feature_matrix_scaled)
cluster_probs = gmm_model.predict_proba(feature_matrix_scaled)
cluster_confidence = cluster_probs.max(axis=1)
cluster_entropy = [entropy(p + 1e-10) for p in cluster_probs]

results_pdf = pd.DataFrame({
    "person_id": features_pdf["person_id"],
    "year": features_pdf["year"],
    "gmm_4_cluster": cluster_labels,
    "gmm_4_confidence": cluster_confidence,
    "gmm_4_entropy": cluster_entropy,
})
results_pdf["gmm_4_density_label"] = results_pdf["gmm_4_cluster"].map(GMM4_LABELS)

# ---- Sanity check: per-cluster feature means ----
profile_pdf = features_pdf[CORE_FEATURES].copy()
profile_pdf["gmm_4_cluster"] = cluster_labels
display(profile_pdf.groupby("gmm_4_cluster").agg(["mean", "count"]))

# ---- Save density scores ----
gmm_output = spark.createDataFrame(results_pdf)
person_window = Window.partitionBy("person_id").orderBy("year")
gmm_output = gmm_output.withColumn("prior_year_gmm_4_cluster", F.lag("gmm_4_cluster", 1).over(person_window))

gmm_output.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
print(f"Wrote {gmm_output.count():,} rows to {OUTPUT_TABLE}")

features.unpersist()

### Step 3 — documentation-volume residuals within each cluster
Writes `cohort_b_derived.cohort_b_edi_residuals`.

In [ ]:
%python
domain_cols = ["total_condition_count", "total_drug_count", "total_procedure_count", "total_measurement_count"]

scored = (
    spark.table("cohort_b_derived.cohort_b_edi_features")
    .join(spark.table(OUTPUT_TABLE).select("person_id", "year", "gmm_4_cluster"), ["person_id", "year"])
)

scored_pdf = scored.select("person_id", "year", "gmm_4_cluster", *domain_cols).toPandas()
scored_pdf[domain_cols] = scored_pdf[domain_cols].fillna(0)

cluster_means = scored_pdf.groupby("gmm_4_cluster")[domain_cols].transform("mean")

for col in domain_cols:
    short = col.replace("total_", "").replace("_count", "")
    scored_pdf[f"domain_resid_{short}"] = scored_pdf[col] - cluster_means[col]
    scored_pdf[f"domain_abs_resid_{short}"] = scored_pdf[f"domain_resid_{short}"].abs()

resid_cols = [c for c in scored_pdf.columns if c.startswith("domain_resid_") or c.startswith("domain_abs_resid_")]
residuals_spark = spark.createDataFrame(scored_pdf[["person_id", "year"] + resid_cols])

residuals_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("cohort_b_derived.cohort_b_edi_residuals")

### Step 4 — join utilization features onto the episode-level feature table
Produces `cohort_b_derived.cohort_b_feature_table_edi`.

In [ ]:
drop table if exists cohort_b_derived.cohort_b_feature_table_edi;
create table cohort_b_derived.cohort_b_feature_table_edi as
WITH episode_year AS (
  SELECT

    e.*,
    add_months(e.ehr_episode_start, 6) AS episode_end_date,   
    YEAR(e.ehr_episode_start) AS start_year,
    DATEDIFF(
      add_months(e.ehr_episode_start, 6), e.ehr_episode_start
    ) AS total_episode_days,
    DATEDIFF(
      LEAST(
        add_months(e.ehr_episode_start, 6),
        DATE_ADD(MAKE_DATE(YEAR(e.ehr_episode_start), 12, 31), 1)
      ),
      e.ehr_episode_start
    ) AS days_in_start_year
  FROM cohort_b_derived.cohort_b_feature_table e
),

gmm_features as (

    select y.ehr_person_id,
    y.ehr_episode_start,
    y.reg_cx_type,
    y.ehr_cx_type,
    y.match_type,
    y.total_contact_days,
    y.years_w_unc,
    y.avg_days_bt_visits,
    y.rad_days,
    y.surg_days,
    y.chemproc_days,
    y.chemdrug_inst,
    y.ccode_days,
    y.ccode_ratio,
    y.treatment_ratio,
    y.start_year,
     gm.gmm_4_cluster
    from episode_year y
    left join cohort_b_derived.gmm_4_density_scores gm
    on y.ehr_person_id = gm.person_id
    and y.start_year = gm.year
)

select f.*, 
e.domain_abs_resid_condition,  
e.domain_abs_resid_drug, 
e.domain_abs_resid_measurement, 
e.domain_abs_resid_procedure,
e.domain_resid_condition, 
e.domain_resid_drug, 
e.domain_resid_measurement, 
e.domain_resid_procedure
FROM gmm_features f
left join cohort_b_derived.cohort_b_edi_residuals e
on f.ehr_person_id = e.person_id and f.start_year = e.year